# Risk, Relative Risk, Odds, and Odds Ratio

**Authors:** Renato Carneiro de Freitas Chaves, Tiago Mendonça dos Santos, Thiago Domingos Corrêa

## 1) Purpose

This  Notebook provides a step-by-step analytical guide for calculating and interpreting four fundamental epidemiological measures:

1. **Risk**
2. **Relative risk**
3. **Odds**
4. **Odds ratio**

The analysis uses a binary exposure variable and a binary outcome variable from the attached dataset.

**Dataset coding**

- `group`: `0 = Control`, `1 = Intervention`
- `outcome`: `0 = Alive`, `1 = Dead`

The event of interest in this report is **death** (`outcome = 1`).

## 2) Required Libraries

This notebook uses standard scientific Python libraries.

In [3]:
import numpy as np
import pandas as pd

## 3) Data Import (Excel)

This guide uses the Excel file **`data.xlsx`** with a sheet named **`Data`**.

- If `data.xlsx` is in the same folder as this notebook, you can keep the path as `"data.xlsx"`.
- Otherwise, replace with the full path to your file.


In [4]:
# Update the path if needed
excel_path = "data.xlsx"
sheet_name = "Data"

data = pd.read_excel(excel_path, sheet_name=sheet_name)

# Quick inspection
data.head(), data.shape

# Column names and basic info
data.info()
list(data.columns)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   patient                 50 non-null     int64         
 1   group                   50 non-null     int64         
 2   center                  50 non-null     int64         
 3   follow_up_start         50 non-null     datetime64[ns]
 4   follow_up_end           50 non-null     datetime64[ns]
 5   follow_up_days          50 non-null     int64         
 6   follow_up_years         50 non-null     float64       
 7   age                     50 non-null     int64         
 8   platelet                50 non-null     int64         
 9   hemoglobin              50 non-null     float64       
 10  creatinine              50 non-null     float64       
 11  hypertension            50 non-null     int64         
 12  diabetes                50 non-null     int64       

['patient',
 'group',
 'center',
 'follow_up_start',
 'follow_up_end',
 'follow_up_days',
 'follow_up_years',
 'age',
 'platelet',
 'hemoglobin',
 'creatinine',
 'hypertension',
 'diabetes',
 'heart_failure',
 'smoking',
 'chronic_kidney_disease',
 'cancer',
 'stroke',
 'saps_3',
 'vasopressor',
 'mechanical_ventilation',
 'fluid_balance',
 'outcome',
 'event']

## 4) Data Validation and Preparation

In [7]:
required_columns = ["group", "outcome"]
missing_columns = [column for column in required_columns if column not in data.columns]

if missing_columns:
    raise ValueError(f"The following required columns are missing: {missing_columns}")

# Frequency checks
data["group"].value_counts(dropna=False), data["outcome"].value_counts(dropna=False)

(1    25
 0    25
 Name: group, dtype: int64,
 1    30
 0    20
 Name: outcome, dtype: int64)

## 5) Create the Analysis Dataset
The analysis dataset keeps only complete observations for `group` and `outcome`.

Definitions:
- **Exposure variable:** `group`
  - `group = 1`: Intervention group
  - `group = 0`: Control group
- **Outcome variable:** `outcome`
  - `outcome = 1`: Death
  - `outcome = 0`: Alive

In [8]:
analysis_data = (
    data[["group", "outcome"]]
    .dropna(subset=["group", "outcome"])
    .copy()
)

# Ensure integer binary coding.
analysis_data["group"] = analysis_data["group"].astype(int)
analysis_data["outcome"] = analysis_data["outcome"].astype(int)

# Create readable labels for tables and interpretation.
analysis_data["group_label"] = analysis_data["group"].map({1: "Intervention", 0: "Control"})
analysis_data["event_label"] = analysis_data["outcome"].map({1: "Death", 0: "Alive"})

# Verify that the expected codes are present.
valid_group_codes = set(analysis_data["group"].unique()).issubset({0, 1})
valid_outcome_codes = set(analysis_data["outcome"].unique()).issubset({0, 1})

if not valid_group_codes or not valid_outcome_codes:
    raise ValueError("The variables 'group' and 'outcome' must be coded as 0/1.")

analysis_data.head()

,group,outcome,group_label,event_label
0,1,0,Intervention,Alive
1,1,0,Intervention,Alive
2,1,0,Intervention,Alive
3,1,0,Intervention,Alive
4,1,0,Intervention,Alive


## 6) Construct the 2×2 Table

Rows:
- Intervention
- Control

Columns:
- Event (Dead)
- Non-event (Alive)

In [9]:
# Create a 2 x 2 table with a stable row and column order.
tab = pd.crosstab(
    analysis_data["group_label"],
    analysis_data["event_label"]
).reindex(
    index=["Intervention", "Control"],
    columns=["Death", "Alive"],
    fill_value=0
)

display(tab)

# Extract the four cell counts.
a = tab.loc["Intervention", "Death"]   # Deaths in the intervention group
b = tab.loc["Intervention", "Alive"]   # Survivors in the intervention group
c = tab.loc["Control", "Death"]        # Deaths in the control group
d = tab.loc["Control", "Alive"]        # Survivors in the control group

# Define group totals.
n_intervention = a + b
n_control = c + d
n_total = n_intervention + n_control

print(f"a = {a}, b = {b}, c = {c}, d = {d}")
print(f"Intervention total = {n_intervention}")
print(f"Control total = {n_control}")
print(f"Total sample size = {n_total}")

event_label,Death,Alive
group_label,,
Intervention,11,14
Control,19,6


a = 11, b = 14, c = 19, d = 6
Intervention total = 25
Control total = 25
Total sample size = 50


## 7. Risk
Risk represents the probability that the event occurs over a defined period in a specified population. In this report, risk refers to the probability of hospital death during the observed hospitalization period.


In [10]:
def safe_divide(numerator, denominator):
    # Return numerator / denominator, or NaN when the denominator is zero.
    return np.nan if denominator == 0 else numerator / denominator

# Calculate the risk of death in each group.
risk_intervention = safe_divide(a, a + b)
risk_control = safe_divide(c, c + d)

# Convert risks to percentages for clinical interpretation.
risk_intervention_percent = risk_intervention * 100
risk_control_percent = risk_control * 100

print(f"Risk of death in the intervention group: {risk_intervention:.4f} ({risk_intervention_percent:.2f}%)")
print(f"Risk of death in the control group: {risk_control:.4f} ({risk_control_percent:.2f}%)")

Risk of death in the intervention group: 0.4400 (44.00%)
Risk of death in the control group: 0.7600 (76.00%)


## 8. Relative Risk

Relative risk, also called the risk ratio, compares the risk of the event in the intervention group with the risk of the event in the control group.

In [11]:
# Calculate the relative risk.
relative_risk = safe_divide(risk_intervention, risk_control)

print(f"Relative risk: {relative_risk:.4f}")

Relative risk: 0.5789


## 9. Odds

Odds express the probability of the event divided by the probability of the event not occurring.

In [12]:
# Calculate the odds of death in each group.
odds_intervention = safe_divide(a, b)
odds_control = safe_divide(c, d)

print(f"Odds of death in the intervention group: {odds_intervention:.4f}")
print(f"Odds of death in the control group: {odds_control:.4f}")

Odds of death in the intervention group: 0.7857
Odds of death in the control group: 3.1667


## 10. Odds Ratio

The odds ratio compares the odds of the event in the intervention group with the odds of the event in the control group.

In [13]:
# Calculate the odds ratio directly from the odds.
odds_ratio = safe_divide(odds_intervention, odds_control)

# Calculate the same odds ratio using the cross-product formula.
odds_ratio_cross_product = safe_divide(a * d, b * c)

print(f"Odds ratio from odds: {odds_ratio:.4f}")
print(f"Odds ratio from cross-product formula: {odds_ratio_cross_product:.4f}")

Odds ratio from odds: 0.2481
Odds ratio from cross-product formula: 0.2481


## 11. Final Summary


In [14]:
final_summary_table = pd.DataFrame({
    "Measure": [
        "Risk of death in intervention group",
        "Risk of death in control group",
        "Relative risk",
        "Odds of death in intervention group",
        "Odds of death in control group",
        "Odds ratio"
    ],
    "Formula": [
        "a / (a + b)",
        "c / (c + d)",
        "[a / (a + b)] / [c / (c + d)]",
        "a / b",
        "c / d",
        "(a / b) / (c / d) = (a × d) / (b × c)"
    ],
    "Estimate": [
        risk_intervention,
        risk_control,
        relative_risk,
        odds_intervention,
        odds_control,
        odds_ratio
    ],
    "Interpretation": [
        "Probability of death among patients in the intervention group.",
        "Probability of death among patients in the control group.",
        "Multiplicative comparison of death risk in the intervention group versus the control group.",
        "Odds of death relative to survival in the intervention group.",
        "Odds of death relative to survival in the control group.",
        "Multiplicative comparison of death odds in the intervention group versus the control group."
    ]
})

final_summary_table["Estimate"] = final_summary_table["Estimate"].round(4)
display(final_summary_table)

,Measure,Formula,Estimate,Interpretation
0,Risk of death in intervention group,a / (a + b),0.4400,Probability of death among patients in the int...
1,Risk of death in control group,c / (c + d),0.7600,Probability of death among patients in the con...
2,Relative risk,[a / (a + b)] / [c / (c + d)],0.5789,Multiplicative comparison of death risk in the...
3,Odds of death in intervention group,a / b,0.7857,Odds of death relative to survival in the inte...
4,Odds of death in control group,c / d,3.1667,Odds of death relative to survival in the cont...
5,Odds ratio,(a / b) / (c / d) = (a × d) / (b × c),0.2481,Multiplicative comparison of death odds in the...


---

## End of Notebook